# Extending the SM by a gauged U(1)$_X$: a Z′ tutorial in `feynlag`

A **Z′ boson** is what you get when the SM gauge group grows an extra abelian
factor U(1)$_X$ [Langacker09]. The SM fermion content (plus three right-handed neutrinos)
admits exactly two independent family-universal anomaly-free abelian charges —
hypercharge $Y$ and $B\!-\!L$ [ADH03] — so the most general consistent assignment is a
linear combination

$$X = a\,Y + b\,(B\!-\!L)$$

with $a$, $b$ free parameters. This notebook keeps them (and the singlet's
charge $q_S$) **symbolic all the way through the pipeline** — declaration,
invariance checking, gauge currents, vertex extraction.

Because the Higgs carries hypercharge, it carries $X$-charge $a/2$, and the
$Z$ and $Z'$ **mix at tree level** — the physically interesting (and
technically instructive) regime. What's new relative to
[`SM_Feynman_Rules_Tutorial.ipynb`](SM_Feynman_Rules_Tutorial.ipynb) (read
that first — `Scalar`, `Model` and the pipeline stages are not re-explained
here; this notebook writes the model at field level with `yukawa`, `tilde`,
`inner` and `add_kinetic`, and lets `model.mass_blocks()` /
`model.diagonalize()` find the physical basis):

- **two independent U(1) factors** in one model (groups are distinct objects;
  a field's `reps` dict just gains a second U(1) entry),
- an **SM-singlet complex scalar** $S$ whose VEV $v_X$ breaks U(1)$_X$, with a
  Higgs-portal coupling $\lambda_{HS}|H|^2|S|^2$ mixing $h$ and $s$,
- **three right-handed neutrinos** (required for the $B\!-\!L$ anomaly to
  cancel) with a Dirac Yukawa via $\tilde H$,
- a **3×3 neutral gauge mass matrix** that the model splits by itself: the
  photon is fixed by the unbroken charge, leaving one $Z$–$Z'$ angle.

The headline formulas we'll derive and verify:

$$\tan 2\theta' = \frac{2\,a\,g_Z g_X v^2}{a^2 g_X^2 v^2 + 4 g_X^2 q_S^2 v_X^2 - g_Z^2 v^2},
\qquad
\tan 2\theta_s = \frac{\lambda_{HS}\, v\, v_X}{\lambda v^2 - \lambda_S v_X^2},
\qquad
m_Z^2\, m_{Z'}^2 = \tfrac{1}{4} g_Z^2 g_X^2 q_S^2 v^2 v_X^2 .$$

$a \to 0$ is the pure gauged-$B\!-\!L$ limit: no mixing, $m_{Z'} = g_X q_S v_X$.

In [ ]:
import sympy as sp

from feynlag import (
    FL_I, FL_J, DiracGamma, Lagrangian, Model, ParameterSet, SU2, SU3,
    Scalar, U1, WeylFermion, conjugate_pair, diracPL, diracPR,
    extract_fermion_vertices, fermion_mass_matrix, inner, rotation_2x2,
    tilde, yukawa,
)

## 1. Symmetries and parameters

Two things distinguish this from the plain-SM setup:

- **A second U(1).** `U1X = U1("U1X", coupling=gX)` — groups compare by
  identity, so `U1Y` and `U1X` coexist as separate keys in every field's
  `reps` dict, each carrying its own coupling and its own gauge boson.
- **Symbolic charges.** `aX`, `bX`, `qS` are ordinary external parameters
  (dimensionless, so the default `unit_dim=0` is right). A U(1) "generator"
  in `feynlag` is just the 1×1 matrix `[[charge]]`, and nothing downstream
  assumes the charge is a number.

Every parameter is declared once with `P.external(...)`/`P.internal(...)`,
which creates it and registers it in the `ParameterSet` handed to the
`Model`.

The charge map `qcharge(Y, B−L) = a·Y + b·(B−L)` below is the entire flavor
structure of the extension — every field's $X$-charge follows from its known
$Y$ and $B\!-\!L$.

In [ ]:
P = ParameterSet()
gw = P.external("g_w", 0.6535, positive=True)
g1 = P.external("g_1", 0.3580, positive=True)
gX = P.external("g_X", 0.5, positive=True)
gs = P.external("g_s", 1.22, positive=True)
SU2L, U1Y = SU2("SU2L", coupling=gw), U1("U1Y", coupling=g1)
U1X = U1("U1X", coupling=gX)
SU3c = SU3("SU3c", coupling=gs)

aX = P.external("a_X", 0.4)      # X = aX*Y + bX*(B-L), kept symbolic
bX = P.external("b_X", -0.7)
qS = P.external("q_S", 1.0)      # singlet X-charge, also symbolic

v = P.external("v", 246.0, positive=True, unit_dim=1)
vX = P.external("v_X", 2000.0, positive=True, unit_dim=1)
lam = P.external("lambda", 0.129)
lamS = P.external("lambda_S", 0.2)
lamHS = P.external("lambda_HS", 0.05)
mu2 = P.internal("mu_2", unit_dim=2)
muS2 = P.internal("mu_{S2}", unit_dim=2)


def qcharge(Y, BmL):
    """The anomaly-free charge map X = a*Y + b*(B-L)."""
    return aX.s * Y + bX.s * BmL

## 2. Fields

Two scalars now:

- the SM Higgs doublet `H`, which picks up $X$-charge $a/2$ (it has $Y=1/2$,
  $B\!-\!L=0$) — **this is what turns on the tree-level $Z$–$Z'$ mixing**;
- an SM-singlet complex scalar `S` with the free charge $q_S$ and VEV $v_X$.

`S` is a one-component complex scalar; `expand_vev` on a complex component
automatically creates the real fluctuation symbols and registers
$S \to (v_X + S0_r + i\,S0_i)/\sqrt{2}$ — same $1/\sqrt2$ convention as the
Higgs. ($S0_r$ is the CP-even "dark Higgs" $s$; $S0_i$ will turn out to be
the Goldstone the $Z'$ eats.)

In [ ]:
H = Scalar("H", reps={SU2L: 2, U1Y: sp.Rational(1, 2),
                      U1X: qcharge(sp.Rational(1, 2), 0)},
           component_names=["Gp", "H0"])
H.expand_vev({H.H0: v})

S = Scalar("S", reps={U1X: qS.s}, component_names=["S0"])
S.expand_vev({S.S0: vX})

S.shift_map     # the automatic complex-VEV expansion

{S0: sqrt(2)*(I*S0_i + S0_r + v_X)/2}

The fermions are the SM's three generations plus **three right-handed
neutrinos** (`nuR`, with $Y=0$, $B\!-\!L=-1$, hence $X$-charge $-b$ — they
carry *only* the new gauge charge). Without them the $B\!-\!L$ part of $X$
would be anomalous; with them, *any* $(a, b)$ is anomaly-free. Every SM
field's `reps` dict simply gains its `U1X: qcharge(...)` entry — nothing
else about the declarations changes.

In [ ]:
Ll = WeylFermion("Ll", reps={SU2L: 2, U1Y: -sp.Rational(1, 2),
                             U1X: qcharge(-sp.Rational(1, 2), -1)},
                 chirality="L", nflavors=3, component_names=["nuL", "eL"])
eR = WeylFermion("eR", reps={U1Y: -1, U1X: qcharge(-1, -1)},
                 chirality="R", nflavors=3, component_names=["eR"])
nuR = WeylFermion("nuR", reps={U1X: qcharge(0, -1)},
                  chirality="R", nflavors=3, component_names=["nuR"])

QL = WeylFermion("QL", reps={SU2L: 2, U1Y: sp.Rational(1, 6), SU3c: 3,
                             U1X: qcharge(sp.Rational(1, 6), sp.Rational(1, 3))},
                 chirality="L", nflavors=3,
                 component_names=["uL_1", "uL_2", "uL_3",
                                  "dL_1", "dL_2", "dL_3"])
uR = WeylFermion("uR", reps={U1Y: sp.Rational(2, 3), SU3c: 3,
                             U1X: qcharge(sp.Rational(2, 3), sp.Rational(1, 3))},
                 chirality="R", nflavors=3,
                 component_names=["uR_1", "uR_2", "uR_3"])
dR = WeylFermion("dR", reps={U1Y: -sp.Rational(1, 3), SU3c: 3,
                             U1X: qcharge(-sp.Rational(1, 3), sp.Rational(1, 3))},
                 chirality="R", nflavors=3,
                 component_names=["dR_1", "dR_2", "dR_3"])

W, B = SU2L.bosons("W"), U1Y.bosons("B")
X = U1X.bosons("X")
G = SU3c.bosons("G")

# components by their declared names (bars: <name>bar), and the canonical
# flavour indices every term constructor uses
i, j = FL_I, FL_J
Gp = H.Gp
nuL, eL, nuLbar, eLbar = Ll.nuL, Ll.eL, Ll.nuLbar, Ll.eLbar
eRc, eRbar = eR.eR, eR.eRbar
nuRc, nuRbar = nuR.nuR, nuR.nuRbar

## 3. The Lagrangian

### Scalar potential with a Higgs portal

$$V = -\mu^2|H|^2 + \lambda|H|^4 - \mu_S^2|S|^2 + \lambda_S|S|^4
      + \lambda_{HS}|H|^2|S|^2$$

The portal term is the only renormalizable, gauge-invariant coupling between
the two scalar sectors — it will mix $h$ and $s$ after both VEVs turn on.
The potential is written with `inner(A, B)` $= A^\dagger B$. The kinetic
terms come from `L.add_kinetic(...)` below, which builds
$(D_\mu\phi)^\dagger D^\mu\phi$ with `Dmu` — and `Dmu` gauges **every**
group in a field's `reps` automatically: for $H$ it includes $W$, $B$ *and*
$X$; for $S$ only $X$.

In [ ]:
HdH, SdS = inner(H, H), inner(S, S)
V = (-mu2 * HdH + lam * HdH**2
     - muS2 * SdS + lamS * SdS**2
     + lamHS * HdH * SdS)

### Yukawas: charged leptons, and neutrinos via $\tilde H$

The charged-lepton Yukawa is the standard
$-Y_e^{ij}\,\bar L_i H\, e_{Rj} + \text{h.c.}$ The **neutrino Dirac Yukawa**
$-Y_\nu^{ij}\,\bar L_i \tilde H\, \nu_{Rj} + \text{h.c.}$ needs the conjugate
doublet $\tilde H = i\sigma_2 H^* = (H^{0*}, -G^{+*})$: that is
`yukawa(Yv, Ll, tilde(H), nuR)`. Each `yukawa` call does the SU(2)
contraction, the flavour indices `Y[fl_i, fl_j]` and the `+ h.c.`, and it
checks the U(1) charge balance on the spot (with the wrong one of
$H$/$\tilde H$ it raises and says which one balances). Check the $X$-charge balance:
$-q_L + q_{\tilde H} + q_{\nu_R} = (a/2 + b) + (-a/2) + (-b) = 0$ — invariant
by construction, for *any* $a, b$. (`check_invariance()` will verify this
symbolically in §4.)

In [ ]:
Ye, Yv = sp.IndexedBase("Ye"), sp.IndexedBase("Yv")
LYuk = yukawa(Ye, Ll, H, eR)            # −Ye L̄ H e_R + h.c.
LYukN = yukawa(Yv, Ll, tilde(H), nuR)   # −Yv L̄ H̃ ν_R + h.c.
LYukN

Gp*conjugate(Yv[fl_i, fl_j])*Bilinear(nuRbar[fl_j], PL, eL[fl_i]) - H0*conjugate(Yv[fl_i, fl_j])*Bilinear(nuRbar[fl_j], PL, nuL[fl_i]) + conjugate(Gp)*Yv[fl_i, fl_j]*Bilinear(eLbar[fl_i], PR, nuR[fl_j]) - conjugate(H0)*Yv[fl_i, fl_j]*Bilinear(nuLbar[fl_i], PR, nuR[fl_j])

The quark Yukawas are the same calls (the colour index is a spectator,
summed diagonally; flavor-generic, no CKM). `add_kinetic` adds the gauge
currents from `fermion_gauge_current`, which, like `Dmu`, is generic over
the `reps` dict, so every fermion's $X$-boson current (with its *symbolic*
charge) appears automatically.

In [ ]:
Yu, Yd = sp.IndexedBase("Yu"), sp.IndexedBase("Yd")
LYuk_d = yukawa(Yd, QL, H, dR)          # −Yd Q̄ H d_R + h.c.
LYuk_u = yukawa(Yu, QL, tilde(H), uR)   # −Yu Q̄ H̃ u_R + h.c.

L = Lagrangian()
L.add_kinetic(H, S, Ll, eR, nuR, QL, uR, dR)   # (Dφ)†(Dφ) + gauge currents
L.add(-V, sector="potential")
L.add(LYuk + LYukN + LYuk_d + LYuk_u, sector="yukawa");

## 4. Assemble the Model and check invariance

This is the first model in `feynlag` with **symbol-valued gauge charges**
going through the invariance checker: the infinitesimal-transformation
residuals now carry factors of $a$, $b$, $q_S$, and invariance means they
cancel *identically in those symbols* — a strictly stronger statement than
for numeric charges (it certifies the whole $(a,b)$ family at once, and
would catch any term that's only invariant for a special charge choice).

In [ ]:
model = Model("SM-U1X", gauge_groups=[SU2L, U1Y, U1X, SU3c],
              fields=[H, S, Ll, eR, nuR, QL, uR, dR, W, B, X, G],
              parameters=P, lagrangian=L)
print(model.summary())

report = model.check_invariance()
report.raise_on_failure()
report

Model: SM-U1X

Fields
field  type       SU2L  U1Y   U1X              SU3c  flavours  components                              VEV
-----  ---------  ----  ----  ---------------  ----  --------  --------------------------------------  -------------
H      scalar     2     1/2   a_X/2            1               Gp, H0                                  ⟨H0⟩ = v/√2
S      scalar     1     0     q_S              1               S0                                      ⟨S0⟩ = v_X/√2
Ll     fermion L  2     -1/2  -a_X/2 - b_X     1     3         nuL, eL
eR     fermion R  1     -1    -a_X - b_X       1     3         eR
nuR    fermion R  1     0     -b_X             1     3         nuR
QL     fermion L  2     1/6   a_X/6 + b_X/3    3     3         uL_1, uL_2, uL_3, dL_1, dL_2, dL_3
uR     fermion R  1     2/3   2*a_X/3 + b_X/3  3     3         uR_1, uR_2, uR_3
dR     fermion R  1     -1/3  -a_X/3 + b_X/3   3     3         dR_1, dR_2, dR_3
W      vector     adj   0     0                1            

InvarianceReport(53 checks, OK)

## 5. EWSB: two tadpoles

Two VEVs, two tadpole conditions, two internal parameters ($\mu^2$,
$\mu_S^2$) to solve for. The portal coupling links them: each $\mu^2$ picks
up a $\lambda_{HS}$-term proportional to the *other* sector's VEV.

In [ ]:
model.solve_tadpoles([mu2, muS2])

{mu_2: (2*lambda*v**2 + lambda_HS*v_X**2)/2,
 mu_{S2}: (lambda_HS*v**2 + 2*lambda_S*v_X**2)/2}

## 6. The scalar sector: portal mixing and Goldstone counting

`model.mass_blocks(kind="scalar")` lists which fluctuations mix (the
connected components of the mass matrix), each with its electric charge
(derived from the vacuum), its CP and its Goldstone directions. The CP-even
block is $(h_0, s_0) = (H0_r, S0_r)$:

In [ ]:
scalar_blocks = model.mass_blocks(kind="scalar")
for block in scalar_blocks:
    print(block)

even = scalar_blocks[0]
h0, s0 = even.fields
even.matrix

scalar  Q=0  cp-even  [H0_r, S0_r]  goldstones: 0
scalar  Q=0  cp-odd  [H0_i]  goldstones: 1
scalar  Q=0  cp-odd  [S0_i]  goldstones: 1
scalar  Q=1  [Gp]  goldstones: 1


Matrix([
[  2*lambda*v**2,   lambda_HS*v*v_X],
[lambda_HS*v*v_X, 2*lambda_S*v_X**2]])

Goldstone counting is a sharp consistency check: two neutral gauge bosons
are about to become massive ($Z$, $Z'$), so exactly two neutral
pseudoscalars must be exactly massless — and indeed $G^0$ and
$\mathrm{Im}\,S$ have no mass term at all after the tadpole substitution
(two 1×1 blocks, each flagged as a Goldstone direction $i T^a\langle\phi\rangle$
of a broken generator), as does the charged Goldstone eaten by $W^\pm$:

In [ ]:
odd = [b for b in scalar_blocks if b.cp == "odd"]
G0, aX0 = (b.fields[0] for b in odd)
Mch = model.mass_matrix([Gp], charged=True)
assert all(b.matrix == sp.zeros(1, 1) and b.goldstones for b in odd)
assert sp.simplify(Mch[0, 0]) == 0
[(b.fields, len(b.goldstones)) for b in odd]   # (fields, # Goldstones)

[([H0_i], 1), ([S0_i], 1)]

`model.diagonalize` on the CP-even block takes the analytic $\tan 2\theta$
route; with a free angle symbol `theta_s` the defining relation is
attached. The names are given light first (`order="light_first"`), so
$h_1 \to h$, the SM Higgs, as $\theta_s \to 0$:

In [ ]:
thetas = sp.Symbol("theta_s", real=True)
d_s = model.diagonalize(even, names=["h1", "h2"], angle=thetas,
                        order="light_first")
h1, h2 = d_s.new_fields
d_s.angle_relation

Eq(tan(2*theta_s), lambda_HS*v*v_X/(lambda*v**2 - lambda_S*v_X**2))

## 7. The neutral gauge sector: one 3×3, one angle

With the Higgs charged under all three neutral generators, the
$(W_3, B, X)$ mass matrix is a genuine 3×3 — the $a$-proportional entries
in the third row/column are the Higgs's doing, and $\det = 0$ promises
exactly one massless photon. `mass_blocks(kind="gauge")` finds it (and the
charged $(W_1, W_2)$ block):

In [ ]:
charged_g, neutral_g = model.mass_blocks(kind="gauge")
print(charged_g)
print(neutral_g)

M3 = neutral_g.matrix
assert sp.simplify(M3.det()) == 0
M3

gauge  Q=1  [W_1, W_2]  massless: 0
gauge  Q=0  [W_3, B, X]  massless: 1


Matrix([
[      g_w**2*v**2/4,    -g_1*g_w*v**2/4,                      -a_X*g_X*g_w*v**2/4],
[    -g_1*g_w*v**2/4,      g_1**2*v**2/4,                       a_X*g_1*g_X*v**2/4],
[-a_X*g_X*g_w*v**2/4, a_X*g_1*g_X*v**2/4, g_X**2*(a_X**2*v**2 + 4*q_S**2*v_X**2)/4]])

There is no symbolic 3×3 eigen-diagonalizer worth using (nested radicals —
see `vacuum/diagonalize.py`'s docstring), and none is needed: the photon
direction is fixed by symmetry alone. The unbroken charge
$Q = \sum_k c_k T_k$ (derived from the vacuum) has the massless vector
$A \propto \sum_k (c_k/g_k)\, A_k$ — here $(g' W_3 + g B)/\sqrt{g^2+g'^2}$,
with no $X$ component, since $S$ carries $X$-charge and a VEV, so $Q$ has
no $X$ piece. `diagonalize` builds the other two directions by Gram–Schmidt
in component order — the SM-like $Z_0 = c_W W_3 - s_W B$ and $X$ — and
diagonalizes the reduced $(Z_0, X)$ block by the same $\tan 2\theta$ route
as the scalars. The result is registered as **one** composed rotation,
verified against $M_3$ (symbolically and numerically) before it is
registered. Names: massive states first, the photon last; with a free
angle symbol `theta_p`, every downstream coupling comes out in terms of
$\theta'$:

In [ ]:
thetap = sp.Symbol("theta_p", real=True)
d_neutral = model.diagonalize(neutral_g, names=["Z", "Zp", "A"], angle=thetap)
Z, Zp, A = d_neutral.new_fields
d_neutral.angle_relation

Eq(tan(2*theta_p), 2*a_X*g_X*v**2*sqrt(g_1**2 + g_w**2)/(a_X**2*g_X**2*v**2 - g_1**2*v**2 + 4*g_X**2*q_S**2*v_X**2 - g_w**2*v**2))

The composed rotation is literally the two-step construction you would do
by hand: **rotation 1 is the standard Weinberg rotation**, untouched by the
extension (it decouples the photon *exactly*, because the unbroken
electric charge doesn't care about $X$), leaving an intermediate $Z_0$ that
still mixes with $X$; **rotation 2** is the $(Z_0, X)$ angle $\theta'$:

In [ ]:
W3, B0, X0 = neutral_g.fields
thetaW = sp.atan(g1.s / gw.s)
cw, sw = sp.cos(thetaW), sp.sin(thetaW)
R1 = sp.Matrix([[cw, -sw, 0], [sw, cw, 0], [0, 0, 1]])  # (W3,B,X) -> (Z0,A,X)
assert sp.simplify(R1 * M3 * R1.T)[1, :] == sp.zeros(1, 3)   # photon row

r2 = rotation_2x2(thetap)                                # (Z0,X) -> (Z,Zp)
R2 = sp.Matrix([[r2[0, 0], 0, r2[0, 1]],
                [r2[1, 0], 0, r2[1, 1]],
                [0, 1, 0]])                              # (Z0,A,X) -> (Z,Zp,A)
assert (d_neutral.rotation.matrix - R2 * R1).applyfunc(sp.simplify) \
    == sp.zeros(3, 3)
d_neutral.rotation.matrix

Matrix([
[ g_w*cos(theta_p)/sqrt(g_1**2 + g_w**2), -g_1*cos(theta_p)/sqrt(g_1**2 + g_w**2), sin(theta_p)],
[-g_w*sin(theta_p)/sqrt(g_1**2 + g_w**2),  g_1*sin(theta_p)/sqrt(g_1**2 + g_w**2), cos(theta_p)],
[              g_1/sqrt(g_1**2 + g_w**2),               g_w/sqrt(g_1**2 + g_w**2),            0]])

## 8. Masses at the benchmark, and the W± rotation

Per `CONVENTIONS.md`, every derived rotation is verified twice, exactly and
numerically — `diagonalize` already did both before registering it. The
benchmark lands where it should — $m_Z \approx 91.6$ GeV, and
$m_{Z'} \approx g_X q_S v_X = 1000$ GeV up to a small mixing correction —
and the photon is exactly massless:

In [ ]:
bench = {gw.s: 0.6535, g1.s: 0.3580, gX.s: 0.5, aX.s: 0.4,
         qS.s: 1.0, v.s: 246.0, vX.s: 2000.0}
th_num = d_neutral.angle_solution.subs(bench)
mZ2, mZp2, mA2 = [float(m.subs(thetap, th_num).subs(bench))
                  for m in d_neutral.masses_squared]
assert mA2 == 0
print(f"m_Z  = {mZ2**0.5:.2f} GeV")
print(f"m_Z' = {mZp2**0.5:.2f} GeV")

m_Z  = 91.62 GeV
m_Z' = 1000.31 GeV


In [ ]:
# the charged block: W± = (W1 ∓ i W2)/√2, read off the unbroken charge
Wp, Wm = model.diagonalize(charged_g, names=["Wp"],
                           conjugate_names=["Wm"]).rotation.new_fields

## 9. Extracting the physical couplings

The boson list now holds **eleven** physical fields — two CP-even scalars,
two pseudoscalars, the charged Goldstones, and five gauge bosons. First a
bosonic highlight from the kinetic sector: the $h_1 Z Z'$ vertex, which
exists *only* because of the mixings (it vanishes as $\theta' \to 0$ with
$\theta_s \to 0$... every factor of it is BSM):

In [ ]:
Gm, cmap = conjugate_pair(Gp, "Gm")
boson_fields = [h1, h2, G0, aX0, Gp, Gm, Z, A, Zp, Wp, Wm]

rules = model.feynman_rules(boson_fields, sector="kinetic",
                            conjugate_map=cmap, simplifier=sp.simplify)
key = tuple(sorted((h1, Z, Zp), key=lambda f: f.sort_key()))
rules[key]

I*(-a_X*g_X*v*(g_1**2 + g_w**2)**3*cos(2*theta_p)*cos(theta_s) - g_1**2*g_w**2*v*(g_1**2 + g_w**2)**(3/2)*(sin(2*theta_p - theta_s) + sin(2*theta_p + theta_s))/2 + g_X**2*(g_1**2 + g_w**2)**(5/2)*(a_X**2*v*cos(theta_s) + 4*q_S**2*v_X*sin(theta_s))*sin(theta_p)*cos(theta_p) - v*(g_1**2 + g_w**2)**(3/2)*(g_1**4 + g_w**4)*(sin(2*theta_p - theta_s) + sin(2*theta_p + theta_s))/4)/(2*(g_1**2 + g_w**2)**(5/2))

### Z′ couplings — the headline

`physical_lagrangian(sector="yukawa")` applies the vacuum shift, both
tadpole solutions, and every registered rotation;
`extract_fermion_vertices` groups the result by `(bar, gamma, field)`.
Every $Z' f f$ coupling should come out as

$$g_{Z'ff} = -\sin\theta'\; g_Z\,(T^3 - Q\,s_W^2)\; +\; \cos\theta'\; g_X\, q_f$$

— the weak-basis $Z_0$ and $X$ couplings reshuffled by the mixing angle.

In [ ]:
LYuk_phys = model.physical_lagrangian(sector="yukawa")
table = extract_fermion_vertices(LYuk_phys, boson_fields)

mu = sp.Symbol("mu", integer=True)
gammaL = DiracGamma(mu) * diracPL
gammaR = DiracGamma(mu) * diracPR


def coeff(bar, gamma, field, boson):
    entry = table.get((bar, gamma, field))
    if entry is None:
        return sp.S.Zero
    return entry.get(1, {}).get((boson,), sp.S.Zero)

In [ ]:
sp.simplify(coeff(eRbar[i], gammaR, eRc[i], Zp))   # Z' e_R e_R

-a_X*g_X*cos(theta_p) - b_X*g_X*cos(theta_p) - g_1**2*sin(theta_p)/sqrt(g_1**2 + g_w**2)

In [ ]:
sp.simplify(coeff(nuRbar[i], gammaR, nuRc[i], Zp))  # Z' nu_R nu_R: pure gX*(-b)*cos

-b_X*g_X*cos(theta_p)

$\nu_R$ has $T^3 = Q = Y = 0$: its whole gauge life is the $X$ charge $-b$,
so its $Z'$ coupling is pure $\cos\theta'\, g_X\,(-b)$ — a Z′ portal to
otherwise-sterile neutrinos.

### Modified Z couplings

The same reshuffle gives every SM fermion a $\sin\theta' \cdot g_X q_f$
correction to its $Z$ coupling ($\theta' \to 0$ recovers the SM — the
practical handle electroweak precision data uses to bound $\theta'$):

In [ ]:
zeR = sp.simplify(coeff(eRbar[i], gammaR, eRc[i], Z))
zeR

-a_X*g_X*sin(theta_p) - b_X*g_X*sin(theta_p) + g_1**2*cos(theta_p)/sqrt(g_1**2 + g_w**2)

In [ ]:
zeR.subs(thetap, 0)    # SM limit: gZ * sw^2

g_1**2/sqrt(g_1**2 + g_w**2)

### Photon couplings — a sanity check

The second rotation touched $Z_0$ and $X$ but never $A$, so photon
couplings are *exactly* the SM electric charges, for any $a, b, \theta'$ —
and the neutrinos (both chiralities) stay dark:

In [ ]:
sp.simplify(coeff(eLbar[i], gammaL, eL[i], A))     # -e, exactly

-g_1*g_w/sqrt(g_1**2 + g_w**2)

In [ ]:
coeff(nuRbar[i], gammaR, nuRc[i], A)               # 0 identically

0

### Higgs-portal couplings

The scalar rotation splits every SM Yukawa coupling between the two mass
eigenstates: $h_1 \bar f f \propto \cos\theta_s$ (the SM value, suppressed)
and $h_2 \bar f f \propto \sin\theta_s$ (how the mostly-singlet scalar
talks to SM fermions at all):

In [ ]:
sp.simplify(coeff(eLbar[i], diracPR, eRc[j], h1))

-sqrt(2)*cos(theta_s)*Ye[fl_i, fl_j]/2

In [ ]:
sp.simplify(coeff(eLbar[i], diracPR, eRc[j], h2))

sqrt(2)*sin(theta_s)*Ye[fl_i, fl_j]/2

### Neutrino Dirac masses

The $\tilde H$ Yukawa delivers a flavor-generic Dirac mass matrix
$M_\nu = Y_\nu\, v/\sqrt2$ (a Majorana $\nu_R$ mass from
$S\,\nu_R^{\mathsf T}C\nu_R$ — i.e. a type-I seesaw — is not included here: it is
only $U(1)_X$-invariant for $q_S=2b$, and this notebook keeps $q_S$ free. The
charge-conjugation machinery it needs, `MajoranaBilinear` with `diracC`, is
walked through in `SM_Seesaw_Tutorial`):

In [ ]:
M_nu = fermion_mass_matrix(LYukN, nuLbar, nuRc, model.vacuum, 3, (i, j),
                           gamma=diracPR)
M_nu

Matrix([
[sqrt(2)*v*Yv[0, 0]/2, sqrt(2)*v*Yv[0, 1]/2, sqrt(2)*v*Yv[0, 2]/2],
[sqrt(2)*v*Yv[1, 0]/2, sqrt(2)*v*Yv[1, 1]/2, sqrt(2)*v*Yv[1, 2]/2],
[sqrt(2)*v*Yv[2, 0]/2, sqrt(2)*v*Yv[2, 1]/2, sqrt(2)*v*Yv[2, 2]/2]])

## 10. Recap

From one charge map $X = aY + b(B\!-\!L)$ and one singlet scalar, `feynlag`
mechanically derived: symbolic-charge gauge invariance of the whole
Lagrangian, two coupled tadpoles, the portal-mixed scalar spectrum with its
two exactly-massless Goldstones, the 3×3 neutral gauge mass matrix split by
the model itself into the photon (fixed by the unbroken charge) and one
$Z$–$Z'$ angle with a closed-form $\tan 2\theta'$, and the full physical coupling table — $Z'ff$ for every
species, the $\theta'$-shifted $Z$ couplings, untouched photon couplings,
and $\cos\theta_s/\sin\theta_s$-split Higgs couplings.

- Exact pinned formulas for everything here (plus the $a \to 0$ B−L limit,
  the W-sector non-renormalization, and the mass invariants):
  [`tests/test_u1x.py`](../../tests/test_u1x.py).
- The full flat script: [`examples/sm_u1x.py`](../../examples/sm_u1x.py).
- The plain-SM baseline:
  [`SM_Feynman_Rules_Tutorial.ipynb`](SM_Feynman_Rules_Tutorial.ipynb).
- Not attempted here: kinetic mixing $\varepsilon B_{\mu\nu}X^{\mu\nu}$
  (no field-strength primitive in the library yet) and UFO export (the UFO
  particle table needs numeric charges — substitute values for $a$, $b$,
  $q_S$ first).

---
## 11. References

- **[Langacker09]** P. Langacker, *The Physics of Heavy Z′ Gauge Bosons*, Rev. Mod. Phys.
  **81** (2009) 1199–1228, doi:[10.1103/RevModPhys.81.1199](https://doi.org/10.1103/RevModPhys.81.1199),
  [arXiv:0801.1345](https://arxiv.org/abs/0801.1345) — Z′ couplings, Z–Z′ mixing and
  anomaly-free U(1)′ models.
- **[ADH03]** T. Appelquist, B. A. Dobrescu, A. R. Hopper, *Nonexotic Neutral Gauge
  Bosons*, Phys. Rev. D **68** (2003) 035012,
  doi:[10.1103/PhysRevD.68.035012](https://doi.org/10.1103/PhysRevD.68.035012),
  [arXiv:hep-ph/0212073](https://arxiv.org/abs/hep-ph/0212073) — with three $\nu_R$, the
  generation-universal anomaly-free U(1)′ charges are exactly $aY+b(B-L)$.